**Large Language Models**

_Copyright Notice_

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Retrieval-Augmented Generation

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

In [2]:
# retrieve API key and other environment variables
load_dotenv()

True

## Setup Vector Store

In [3]:
from langchain_chroma import Chroma

In [4]:
from langchain_openai import OpenAIEmbeddings

In [5]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [6]:
db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model
)

## Retrieve

In [7]:
# MMR(Maximal Marginal Relevance)
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5, "lambda_mult": 0.8}
)

In [8]:
question = "노아의 홍수는 진짜 있었던 일인가요?"

In [9]:
context = retriever.invoke(question)

In [10]:
context[0].metadata

{'moddate': "D:20241209114208Z00'00'",
 'page': 6,
 'page_label': '7',
 'source': './data/세계의-홍수-전설.pdf',
 'creator': 'Word',
 'author': 'Jaeyoung Chun',
 'producer': 'macOS Version 14.5 (Build 23F79) Quartz PDFContext',
 'creationdate': "D:20241209114208Z00'00'",
 'total_pages': 7}

## Augment

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [12]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="당신은 지적설계를 지지하는 과학자입니다. 당신의 청중은 대학생입니다."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            {question}
            아래의 문맥에 기반하여 답해주세요.
            {context}
            """
        )
    ]
)

In [13]:
message = chat_template.format_messages(
    question=question,
    context=context[0].page_content
)

## Generate

In [14]:
from langchain_openai import ChatOpenAI

In [15]:
model = ChatOpenAI(
    model_name="gpt-6-luna",
)

In [16]:
for chunk in model.stream(message):
    print(chunk.content, end="", flush=True)

**문맥의 주장은 “홍수 전설들이 전 세계에 퍼져 있다는 점은 실제 대홍수의 기억이 보존된 결과일 수 있다”는 해석입니다.** 노아와 그 후손이 홍수와 방주 이야기를 전했고, 각 지역에서 구전되며 조금씩 달라졌다는 설명이지요. 이는 성경을 역사적 기록으로 읽는 관점에서는 일관된 해석입니다.

다만 **홍수 전설의 유사성만으로 노아의 홍수나 전 지구적 홍수가 과학적으로 입증되었다고 말하기는 어렵습니다.** 홍수 이야기는 큰 강이나 해안 가까이 살았던 여러 사회에서 실제로 겪은 지역적 재난을 바탕으로 생겨날 수도 있고, 이야기의 전파나 공통된 서사 유형으로 비슷해질 수도 있습니다. 피지의 이야기가 방주를 카누처럼 묘사한다는 사례도 흥미로운 비교이지만, 그것만으로 성서 이야기에서 직접 전해졌다고 입증되지는 않습니다.

현재 지질학·고생물학의 주류 증거는 **최근 인류 역사 시기에 지구 전체가 한 차례의 홍수로 잠겼다는 주장과 맞지 않습니다.** 전 지구를 동시에 덮은 홍수에 해당하는 단일 지질층이 확인되지 않고, 암석층과 화석 기록은 오랜 시간에 걸쳐 형성된 것으로 해석됩니다. 반면 메소포타미아 등에서는 큰 강의 범람이 실제로 일어났으며, 그런 재난이 지역 전승과 고대 홍수 이야기의 배경이 되었을 가능성은 있습니다.

따라서 질문에 정확히 답하면, **성경과 제시된 문맥의 해석에 따르면 노아의 홍수는 실제 사건입니다. 그러나 세계 각지의 홍수 전설은 그 해석을 뒷받침하는 단서일 수는 있어도, 전 지구적 홍수를 과학적으로 증명하는 증거로 받아들여지지는 않습니다.** 노아 이야기의 역사성을 믿는 신앙적 판단과, 지질학 자료가 지지하는 범위를 구분해 보는 것이 좋습니다.